# 00 — Subtask 1 Winning Strategy

*Last updated: 2026-05-04 — 3 days to deadline (May 7).*

This notebook is the **master strategy document**. It builds intuition,
catalogues every technique worth trying, and makes opinionated decisions
about where to spend the 72 hours and $75 RunPod budget.

Read this first. Each section maps to specific implementation notebooks.

## TL;DR — what this strategy commits to

1. **Domain features beat raw bands** — add 7 vegetation indices + phenological
   metrics before anything else. NB6.
2. **Ordinal head over flat classification** — switch the U-Net to CORN
   (Conditional Ordinal Regression for Neural networks) to optimise Accuracy±1
   directly. NB7.
3. **Pseudo-labelling unlocks the 800 unlabeled test patches** — confidence-thresholded
   self-training, 2 rounds. NB9.
4. **Diverse ensemble of 5 models** — LightGBM + 2 U-Nets (different temporal
   reductions) + CORN U-Net + pseudo-trained U-Net. NB10.
5. **Calibration trick exploits Accuracy±1** — bias predictions toward the
   middle class when uncertain; round expected-value rather than argmax.
6. **D4 TTA + 5×5 median filter** at inference — free 1–3% gains.

**Expected leaderboard journey:** 0.70 (majority class) → 0.82 (NB2 tabular) →
0.88 (NB3 U-Net) → 0.90 (CORN + features) → 0.92+ (full ensemble).

## 1. Problem reframe — what is the task *really* asking?

AgriPotential isn't classification. It's **terroir mapping**: how suitable is
this 5×5m square for growing wine grapes? The 5 ordinal classes (very low → very
high) reflect agronomic potential, which depends on:

- **Vegetation vigour pattern over the year** — vineyards have a distinct
  phenology (bud-break in spring, peak in summer, leaf-off in autumn) very
  different from corn, wheat, fallow.
- **Water availability** — visible in NIR/SWIR reflectance during dry season.
- **Soil exposure during winter** — bare-soil periods reveal organic matter,
  texture, drainage.
- **Stability across timeframes** — high-potential land has consistent
  vegetation; marginal land flickers.

This means the predictive signal lives in **temporal patterns** more than in
any single timestep. A model that ignores time (just averages bands) will leave
a lot on the table.

**Implication:** features that summarise the *shape* of the time series
(growing-season length, peak day, integrated NDVI, harmonic decomposition) are
high-leverage. Vanilla per-timeframe statistics aren't enough.

---

### Data shape recap

| Item | Value |
|---|---|
| Train patches | 6,329 |
| Val patches | 781 |
| Test patches | 800 (labels hidden) |
| Patch size | 128 × 128 px (5 m/px) |
| Timeframes | 34 Sentinel-2 acquisitions, 2017–2019 |
| Bands | 10 (B1–B8, B8A, B9; no SCL/QA) |
| Label | 5 ordinal classes (0–4), per pixel |
| Submission | ZIP of 800 PNGs at root, `<patch_id>.png`, uint8 0–4 |

## 2. Why Accuracy±1 changes everything

Accuracy±1 = fraction of pixels where `|pred - truth| ≤ 1`. This is **not**
ordinary classification accuracy. Three implications govern the entire strategy:

### 2.1 Mistakes are not symmetric — being far is catastrophic

| True | Predicted | Reward |
|---|---|---|
| 2 | 2 | ✓ exact |
| 2 | 1 or 3 | ✓ Acc±1 |
| 2 | 0 or 4 | ✗ wrong |

A confident prediction of class 0 when truth is class 2 is the **same loss**
as predicting class 4 when truth is class 0. There's no partial credit beyond ±1.

### 2.2 Optimal prediction under uncertainty is the *expected value*, rounded

If you have probability distribution `p[0..4]`, the prediction that maximises
Accuracy±1 in expectation is the **median** of the distribution (the smallest k
such that `sum(p[0..k]) ≥ 0.5`), not the argmax. Equivalently, for symmetric
distributions the rounded expected value `round(sum(k * p[k]))` is optimal.

When `p = [0.3, 0.05, 0.25, 0.05, 0.35]` (bimodal), argmax says class 4,
but expected value rounds to class 2, which is closer to either mode.
**This single change can be worth 1–3% Acc±1 on noisy predictions.**

### 2.3 Spatial smoothing is essentially free

Adjacent pixels share class with high probability. A 5×5 median filter on the
predicted mask removes salt-and-pepper noise without blurring meaningful
boundaries. Because the metric tolerates ±1 error, even slightly mis-aligned
smoothing usually still scores correct.

### 2.4 Training loss should match the metric

Cross-entropy treats class 0 and class 4 as equally distant from class 2. **MSE
on class index** or **CORN ordinal heads** treat them properly. This is
non-negotiable — almost every winning solution to ordinal-metric competitions
uses an ordinal-aware loss.

## 3. Catalogue of techniques — every option under the sun

Each row: name, what it does, why it helps for *this* problem, expected gain,
effort. Gains are stacked relative to the previous tier (not all additive).

### Tier A — Must Have (high leverage, low effort)

| Technique | Why | Gain | Effort |
|---|---|---|---|
| Vegetation indices (NDVI, NDRE, NDWI, EVI, SAVI, GCVI, PRI) | Captures vegetation/water/chlorophyll; viticulture is a vegetation problem | +5–8% | 1 hr |
| Phenological metrics (peak NDVI day, integrated NDVI, growing-season length) | Encodes time-series *shape* in compact features | +2–4% | 2 hr |
| Ordinal soft labels or CORN head | Loss matches metric | +2–3% | 2 hr |
| Class-balanced sampling | Avoids majority-class bias | +1–2% | 30 min |
| Median filter post-processing (5×5) | Removes pixel noise; almost free Acc±1 | +1–2% | 5 min |
| D4 TTA (8 augmentations) | Cheap inference-time ensemble | +1–2% | 30 min |
| Expected-value prediction (vs argmax) | Acc±1-optimal decoder | +1–3% | 5 min |

**Tier A alone should get you from 0.82 → 0.90+ Accuracy±1.**

### Tier B — High Leverage, Medium Effort

| Technique | Why | Gain | Effort |
|---|---|---|---|
| Multi-architecture ensemble (LightGBM + U-Net + CORN U-Net) | Different inductive biases reduce correlated errors | +2–3% | 4 hr |
| Pseudo-labelling on test patches | 800 extra patches × 16K pixels each = 13M new training samples | +1–3% | 4 hr |
| Snapshot ensemble (cosine warm restarts, average N checkpoints) | Free improvement; no extra training time | +0.5–1% | 1 hr |
| Stratified spatial K-fold CV | Better val estimate, average K models | +1–2% | 4 hr |
| Savitzky-Golay temporal smoothing on NDVI | Removes cloud spikes before computing phenology | +0.5–1% | 1 hr |
| Calibration on val (temperature scaling + class-prior adjustment) | Sharpens probabilities for correct expected-value decoding | +0.5–1% | 1 hr |

**Tier B layered on Tier A: 0.90 → 0.92+.**

### Tier C — Worth Trying if Time Permits

| Technique | Why | Gain | Effort |
|---|---|---|---|
| L-TAE / PSE+TAE temporal attention | SOTA for satellite time series; pixel-level attention over time | +1–2% | 8 hr |
| Self-supervised MAE pretraining on test patches | Leverage 800 unlabeled patches for representation learning | +1% | 12 hr |
| Mixup / CutMix augmentation | Regularisation for over-fitting | +0.5–1% | 2 hr |
| TSViT (Temporal-Spatial ViT) | SOTA 2023+ for satellite seg; complex | +1–2% | 16 hr |
| CRF / DenseCRF post-processing | Edge-aware smoothing | +0.3–0.7% | 4 hr |

### Tier D — Skip (low gain or too risky for 3 days)

- Diffusion-based segmentation
- Multi-scale prediction (we have fixed patch size)
- Foundation-model fine-tuning (Prithvi, SatlasNet) — setup cost too high
- Radiative-transfer corrections (data is L2A already)
- AdaIN/style transfer for domain adaptation

## 4. The decision — 3-day execution plan

I'm committing to this hour-by-hour plan. It assumes RunPod uptime (≈$0.71/hr,
$75 budget = ~105 hr ceiling, so we have slack).

### Day 1 (today, May 4): get to a strong baseline

| Hours | Action | Notebook | Goal |
|---|---|---|---|
| 0–2 | Run NB1 EDA on RunPod | `01_eda` | Confirm class distribution, identify discriminative timeframes |
| 2–4 | Run NB6 advanced features (vegetation indices + phenology) | `06_advanced_features` | Cache extended feature set |
| 4–8 | Run NB2 pixel baseline with extended features | `02_pixel_baseline` | Target val Acc±1 ≥ 0.85 |
| 8–10 | **Submit Day-1 baseline ZIP to CodaBench** — establishes leaderboard floor | `05_inference_submit` | Verify pipeline end-to-end |

### Day 2 (May 5): deep models + diversity

| Hours | Action | Notebook | Goal |
|---|---|---|---|
| 10–18 | Train U-Net `summary` variant | `03_unet` | Val Acc±1 ≥ 0.88 |
| 18–28 | Train CORN ordinal U-Net | `07_corn_ordinal_unet` | Val Acc±1 ≥ 0.89, save predictions |
| 28–36 | Train U-Net `seasonal` variant for diversity | `03_unet` (re-run) | Different errors than `summary` |

### Day 3 (May 6): pseudo-labelling + ensemble + submit

| Hours | Action | Notebook | Goal |
|---|---|---|---|
| 36–48 | Generate pseudo-labels and re-train top model | `09_pseudo_labeling` | Val Acc±1 ≥ 0.90 |
| 48–60 | Stacked ensemble + calibration + median filter + TTA | `10_final_stack` | Val Acc±1 ≥ 0.92 |
| 60–66 | **Submit final ZIP** to CodaBench, save model artifacts | `10_final_stack` | Locked submission |
| 66–72 | Buffer for re-runs / leaderboard feedback / docs | — | Slack |

### Day 4 (May 7): submission deadline morning

Only run if leaderboard feedback says we're not in the top 5: try one
experimental change (e.g., different median filter size, different α weight)
and submit one alternative.

**Stop the RunPod Pod whenever idle.** Cost discipline > model perfection.

## 5. Why this beats the median competitor

Most teams will:

1. Train one U-Net or one tabular model.
2. Use cross-entropy loss with hard labels.
3. Submit argmax predictions.
4. Maybe add a median filter as an afterthought.

Our strategy stacks **eight independent improvements** on top of that:

1. Domain-specific features (vegetation indices, phenology) → captures actual
   signal
2. CORN ordinal head → loss matches metric
3. Pseudo-labelling → unlocks 800 unlabeled test patches
4. Multi-architecture ensemble (5 models) → reduces correlated errors
5. Stacked weights tuned on val → optimal blend, not arithmetic mean
6. Expected-value decoding → metric-optimal, not argmax
7. D4 TTA → 8× inference, 1–2% boost
8. 5×5 median filter → spatial coherence

Each of these is well-known on its own. The win is **stacking all of them**
in 3 days. Most teams will get 3–4 of these; few will get all 8.

## 6. Validation methodology — how do we know we're winning?

### 6.1 Val Acc±1 progression — track in `results/runs.csv`

After each training run, append a row with: timestamp, owner, notebook,
model name, val Acc±1, val exact accuracy, artifact path, notes.

### 6.2 The single most important plot — learning curve over runs

Plot val Acc±1 against cumulative GPU hours. If the slope flattens before
you've used 60% of your hours, your bottleneck is no longer compute — it's
ideas. That's when you reach for Tier C techniques.

### 6.3 Confusion matrices — read every one

Every model dumps its val confusion matrix. Look for:

- **Off-diagonal mass at distance > 1** — these are the catastrophic errors
  that hurt Acc±1 most. Increase loss weight on these.
- **Class 0 / class 4 recall** — usually the weakest, since they're rare and
  visually similar to neighbours. Class-balanced sampling and edge weights help.
- **Asymmetry** — if model under-predicts class 4 (predicts class 3 instead),
  that's actually fine for Acc±1 but reveals a learnable correction.

### 6.4 Don't trust val Acc±1 if it improves > 0.5% per submission

Sustained gains > 0.5% from minor changes usually mean over-fitting to val.
Cross-check by holding out a few hundred train patches as a second val set.

## 7. Risk register

| Risk | Likelihood | Mitigation |
|---|---|---|
| RunPod runs out of quota / Pod restarts | Medium | Save checkpoints every epoch; sync to local every few hours |
| HF streaming bandwidth too slow | High | Pre-download dataset to `/workspace/ai4agri/data/subtask1` |
| OOM on full-size U-Net | Low | Batch size 4 fits with `concat` mode (340 channels); fallback `summary` |
| CodaBench submission limit blocks final submit | Unknown | Submit Day 1 baseline + Day 2 mid + Day 3 final → at least 1 confirmed valid |
| Pseudo-labels degrade model | Medium | Threshold confidence ≥ 0.85; only iterate if val Acc±1 doesn't drop |
| Val/test domain shift | Medium | If submission Acc±1 ≪ val Acc±1, switch to less-fit model |
| Time runs out | Medium | Day-3 final ensemble is the floor; Day-1 baseline is the absolute backup |

## 8. Quick experiments — building intuition before committing

The cells below run instantly (read CSV metadata only) to ground three intuitions.
Before training anything, confirm these match your prior.

In [3]:
# Quick: what's the majority-class baseline Acc±1?
# If class 2 dominates, just predicting class 2 for every pixel scores ≈ fraction(class 1+2+3).
# This is our absolute floor. A trained model below this is broken.

import csv, os
from pathlib import Path
from urllib.parse import urljoin
from urllib.request import urlopen

_cwd = Path(os.getcwd())
REPO_ROOT = "/workspace/ai4agri" #_cwd if (_cwd / "scripts").exists() else _cwd.parent
REPO_ROOT = Path(REPO_ROOT)
#HF = "https://huggingface.co/datasets/m-sakka/agripotential/resolve/main/"
_local = REPO_ROOT / "data" / "subtask1"
DATA_DIR = str(_local) if _local.exists() else None  # auto-detect local copy

def ref(name):
    return str(Path(DATA_DIR)/name) if DATA_DIR else urljoin(HF, name)

def csv_rows(name):
    r = ref(name)
    if r.startswith("http"):
        with urlopen(r, timeout=60) as h: lines = [l.decode() for l in h]
    else:
        lines = Path(r).read_text().splitlines()
    return list(csv.DictReader(lines))

print(f"DATA_DIR: {DATA_DIR or '(HuggingFace remote)'}")
metadata = csv_rows("metadata.csv")
train    = csv_rows("train.csv")
val      = csv_rows("val.csv")
test     = csv_rows("test.csv")
print(f"Train/Val/Test patches: {len(train)} / {len(val)} / {len(test)}")
print(f"Timeframes: {len(metadata)}  Date columns: {list(metadata[0].keys())}")
print(f"Patch columns: {list(train[0].keys())}")


DATA_DIR: /workspace/ai4agri/data/subtask1
Train/Val/Test patches: 6329 / 781 / 800
Timeframes: 34  Date columns: ['filename', 'day', 'month', 'year']
Patch columns: ['patch_id', 'row', 'col', 'patch_size', 'n_annotated']


In [4]:
# Compute the *theoretical* Acc±1 for predicting always class 2.
# Requires reading some labels; we approximate with NB1's class_distribution.png if available.
# Otherwise this just shows the formula and a placeholder.

import json
summary_path = REPO_ROOT / "results" / "subtask1" / "inspection" / "eda_summary.json"
if summary_path.exists():
    s = json.loads(summary_path.read_text())
    if "class_pixel_fractions" in s:
        f = s["class_pixel_fractions"]
        majority_floor = float(f.get("1", 0)) + float(f.get("2", 0)) + float(f.get("3", 0))
        print(f"Class fractions (from NB1):")
        for k in range(5):
            print(f"  Class {k}: {float(f.get(str(k), 0)):.3f}")
        print(f"\nAcc±1 floor by predicting class 2 everywhere: {majority_floor:.3f}")
        print("Any trained model must beat this. If not, the model is broken.")
    else:
        print("NB1 hasn't run yet. Once it has, re-run this cell to see the floor.")
else:
    print("NB1 hasn't run yet. Run notebooks/01_eda_subtask1.ipynb first.")

NB1 hasn't run yet. Run notebooks/01_eda_subtask1.ipynb first.


In [5]:
# Intuition: timeframe coverage by month — which seasons are over-represented?
# Vineyards mostly differentiate from corn/wheat in summer (peak vigour)
# and winter (bare soil shows). If the dataset is summer-heavy, raw band
# averages already discriminate well. If it's winter-heavy, phenology features
# become essential.

from collections import Counter
month_counts = Counter(int(m["month"]) for m in metadata)
print("Timeframes per month:")
for m in range(1, 13):
    cnt = month_counts.get(m, 0)
    bar = "█" * cnt
    print(f"  {m:2d}: {cnt}  {bar}")
print(f"\nTotal timeframes: {sum(month_counts.values())}")
print("\nIf one season dominates, weight features from the under-represented seasons higher")
print("(they're rarer signals but contain unique discriminative info).")

Timeframes per month:
   1: 3  ███
   2: 1  █
   3: 2  ██
   4: 2  ██
   5: 1  █
   6: 3  ███
   7: 5  █████
   8: 4  ████
   9: 3  ███
  10: 5  █████
  11: 2  ██
  12: 3  ███

Total timeframes: 34

If one season dominates, weight features from the under-represented seasons higher
(they're rarer signals but contain unique discriminative info).


## 9. Notebook map — what runs where

| Notebook | Stage | Purpose |
|---|---|---|
| `00_strategy.ipynb` | Plan | This document |
| `01_eda_subtask1.ipynb` | Day 1 | EDA — class dist, temporal profiles, band correlation |
| `02_pixel_baseline_subtask1.ipynb` | Day 1 | LightGBM/HistGB baseline on per-pixel features |
| `03_unet_subtask1.ipynb` | Day 2 | U-Net with ordinal soft labels |
| `04_ensemble_postprocess_subtask1.ipynb` | Day 3 | Pixel + U-Net ensemble grid search |
| `05_inference_submit_subtask1.ipynb` | Day 1 + Day 3 | Test inference → ZIP → validate |
| `06_advanced_features.ipynb` | Day 1 | Vegetation indices + phenology (NEW) |
| `07_corn_ordinal_unet.ipynb` | Day 2 | CORN ordinal head U-Net (NEW) |
| `09_pseudo_labeling.ipynb` | Day 3 | Pseudo-label + re-train (NEW) |
| `10_final_stack.ipynb` | Day 3 | Stacked ensemble + calibration + submission (NEW) |

(Notebook 8 — temporal attention encoder — reserved for Tier C if time allows.)

Run order:

```text
01 → 06 → 02 → 05 (Day 1 floor submission)
      ↓
  03, 07 in parallel
      ↓
  09 (using best of 03/07)
      ↓
  10 (final stack and submission)
```

## 10. Open questions for VB

Need confirmation before Day 3 submission:

1. **CodaBench submission limits** — how many submissions per day? Per
   competition phase? This determines whether we can iterate on the leaderboard.
2. **Score visibility** — does CodaBench show test Acc±1 immediately, or
   delayed? If immediate, use Day 2 to probe; if delayed, trust val.
3. **Tie-breaking** — if multiple participants score identically, what wins?
4. **Optional report.pdf** — does including it influence ranking, or only used
   for paper review? If only paper, skip until after submission.

These are listed as `[ ]` in `Next.md` and should be answered before May 5.
